# Lesson 16: Clustering

*ML Academy — companion notebook*

Group data without labels: k-means, choosing k, hierarchical clustering and DBSCAN.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-16.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 10. In Python

In [1]:
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

X, y_true = make_blobs(n_samples=500, centers=4, cluster_std=1.0, random_state=42)
for k in range(2, 8):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    print(f"k={k}  inertia={km.inertia_:8.1f}  silhouette={silhouette_score(X, km.labels_):.3f}")

k=2  inertia= 15737.1  silhouette=0.596
k=3  inertia=  3426.3  silhouette=0.761
k=4  inertia=   948.9  silhouette=0.791
k=5  inertia=   855.9  silhouette=0.665
k=6  inertia=   768.7  silhouette=0.561
k=7  inertia=   681.0  silhouette=0.442


In [2]:
Xm, ym = make_moons(n_samples=300, noise=0.06, random_state=0)
Xm = StandardScaler().fit_transform(Xm)

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Xm)
db = DBSCAN(eps=0.3, min_samples=5).fit(Xm)
ag = AgglomerativeClustering(n_clusters=2, linkage="single").fit(Xm)

print("k-means ARI:", round(adjusted_rand_score(ym, km.labels_), 3))
print("DBSCAN  ARI:", round(adjusted_rand_score(ym, db.labels_), 3),
      " clusters:", len(set(db.labels_) - {-1}), " noise points:", list(db.labels_).count(-1))
print("single-linkage ARI:", round(adjusted_rand_score(ym, ag.labels_), 3))

k-means ARI: 0.47
DBSCAN  ARI: 0.993  clusters: 2  noise points: 1
single-linkage ARI: 1.0


## 13. Going deeper: mixture models, density and hierarchy

In [3]:
import numpy as np
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, silhouette_score

# Stretched clusters: k-means assumes round blobs, a Gaussian mixture doesn't
Xb, yb = make_blobs(n_samples=600, centers=3, random_state=170)
Xs = Xb @ np.array([[0.6, -0.6], [-0.4, 0.8]])          # stretch and shear the blobs
km = KMeans(3, n_init=10, random_state=0).fit_predict(Xs)
gmm = GaussianMixture(3, covariance_type="full", random_state=0).fit(Xs)
print("stretched blobs: ARI k-means", round(adjusted_rand_score(yb, km), 3), "  ARI Gaussian mixture", round(adjusted_rand_score(yb, gmm.predict(Xs)), 3))
p = gmm.predict_proba(Xs)
print("share of points the mixture is unsure about (max probability < 0.9):", round(np.mean(p.max(axis=1) < 0.9), 3))

stretched blobs: ARI k-means 0.58   ARI Gaussian mixture 1.0
share of points the mixture is unsure about (max probability < 0.9): 0.002


In [4]:
Xm, ym = make_moons(n_samples=500, noise=0.06, random_state=0)
results = {
    "k-means":            KMeans(2, n_init=10, random_state=0).fit_predict(Xm),
    "Gaussian mixture":   GaussianMixture(2, random_state=0).fit(Xm).predict(Xm),
    "DBSCAN":             DBSCAN(eps=0.2, min_samples=5).fit_predict(Xm),
    "agglomerative, single linkage": AgglomerativeClustering(2, linkage="single").fit_predict(Xm),
    "agglomerative, Ward linkage":   AgglomerativeClustering(2, linkage="ward").fit_predict(Xm),
}
for name, lab in results.items():
    print(f"{name:30}: ARI {adjusted_rand_score(ym, lab):.3f}")

k-means                       : ARI 0.253
Gaussian mixture              : ARI 0.495
DBSCAN                        : ARI 1.000
agglomerative, single linkage : ARI 1.000
agglomerative, Ward linkage   : ARI 0.559


In [5]:
# Choosing k without labels: silhouette score and the mixture's BIC
Xk, _ = make_blobs(n_samples=600, centers=4, cluster_std=1.0, random_state=7)
print(" k   silhouette   BIC")
for k in range(2, 8):
    lab = KMeans(k, n_init=10, random_state=0).fit_predict(Xk)
    bic = GaussianMixture(k, random_state=0).fit(Xk).bic(Xk)
    print(f"{k:2d}   {silhouette_score(Xk, lab):10.3f}   {bic:7.0f}")

 k   silhouette   BIC
 2        0.588      6194
 3        0.762      5490
 4        0.811      5133


 5        0.680      5168


 6        0.577      5193


 7        0.446      5227


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [6]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
cols = ["size_m2", "dist_km", "price_lakh"]
Z = StandardScaler().fit_transform(df[cols])
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
df["cluster"] = km.labels_
print(df.groupby("cluster")[cols].mean().round(1))
print(pd.crosstab(df["cluster"], df["area"]))

         size_m2  dist_km  price_lakh
cluster                              
0           82.0      6.2        66.2
1          129.0     11.0        88.4
2          112.5     22.5        46.0
area     Central  Outskirts  Suburb
cluster                            
0              3          0       2
1              1          1       3
2              0          2       0


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [7]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: The assign step

Give each point the index of its nearest centroid (1D).

In [8]:
import numpy as np
x = np.array([1, 2, 6, 7, 8.0])
centroids = np.array([1.0, 8.0])
labels = None  # ← your code here
check(labels, [0, 0, 1, 1, 1], hint='Build a points × centroids table of distances, then argmin along axis=1.')

❌ Not yet. Hint: Build a points × centroids table of distances, then argmin along axis=1.


### Challenge 2: The update step

Move each centroid to the mean of its points.

In [9]:
import numpy as np
x = np.array([1, 2, 6, 7, 8.0])
labels = np.array([0, 0, 1, 1, 1])
new_centroids = None  # ← your code here
check(new_centroids, [1.5, 7.0], hint='For each cluster k, average x[labels == k].')

❌ Not yet. Hint: For each cluster k, average x[labels == k].


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. The assign step** — the missing line:

```python
labels = np.argmin(np.abs(x[:, None] - centroids[None, :]), axis=1)
```

**2. The update step** — the missing line:

```python
new_centroids = np.array([x[labels == k].mean() for k in range(2)])
```